In [11]:
# VAR robustness (Section 4.8): Table 6, Tables B.2, C.3, C.4 and C.7, Section 5.1
import os
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.var_model import VAR

warnings.filterwarnings("ignore")
os.makedirs("output", exist_ok=True)

In [12]:
# Merge the clean files into one monthly data set, December 2015 to December 2025 (121 months)
files = ["spy_clean.csv", "rsp_clean.csv", "voo_clean.csv", "vix_clean.csv", "dgs10_clean.csv",
         "AIStocks_ASVI8_Monthly.csv", "AIStocks_ASVI4_Monthly.csv",
         "mag7_clean.csv", "mag7b_clean.csv", "mag7c_clean.csv"]
df = pd.read_csv(files[0], parse_dates=["date"])
for f in files[1:]:
    df = df.merge(pd.read_csv(f, parse_dates=["date"]), on="date", how="inner")
df = df.sort_values("date").reset_index(drop=True)

# 2021-2025 subsample. December 2020 only serves as the first lag, so the VARs run on Jan 2021 to Dec 2025.
sub = df[df["date"] >= "2020-12-31"].reset_index(drop=True)
print(len(df), "months,", df["date"].min().date(), "to", df["date"].max().date())

labels = {"dlog_vix": "dlogVIX", "d_dgs10": "dDGS10", "ASVI8": "ASVI8", "ASVI4": "ASVI4",
          "flow_spy": "Flow SPY", "flow_rsp": "Flow RSP", "flow_voo": "Flow VOO",
          "mag7_ret": "Mag7 returns", "mag7b_ret": "Mag7B returns", "mag7c_ret": "Mag7C returns"}

samples = {"Full sample": df, "2021-2025": sub}

121 months, 2015-12-31 to 2025-12-31


In [13]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""

def granger(res, cause, effect):
    # F-test: do the lags of `cause` help predict `effect`?
    t = res.test_causality(effect, [cause], kind="f")
    return float(t.test_statistic), float(t.pvalue)


def fevd(res, response):
    # Share (%) of the forecast error variance of `response` explained by each shock, h = 1, 6, 12
    d = res.fevd(12).decomp[list(res.names).index(response)] * 100
    return pd.DataFrame({"h=1": d[0], "h=6": d[5], "h=12": d[11]}, index=list(res.names))


def irf_bands(res, impulse, response, h=12):
    # Orthogonalized impulse response with asymptotic 95% bands
    i, j = list(res.names).index(response), list(res.names).index(impulse)
    irf = res.irf(h)
    point = irf.orth_irfs[:, i, j]
    se = irf.stderr(orth=True)[:, i, j]
    return point, point - 1.959964 * se, point + 1.959964 * se


def largest_eigenvalue(res):
    # statsmodels reports the roots of the characteristic polynomial. The eigenvalues of the
    # companion matrix are their inverses, so the largest eigenvalue is 1 / smallest root.
    return 1 / np.abs(res.roots).min()

In [14]:
# 22 specifications, each estimated on both samples:
#   A: Mag7 with the SPY, RSP or VOO flow (4.8.1)
#   B: Mag7B or Mag7C with each flow (4.8.2)
#   C: three-variable VAR without the yield and the flow (4.8.3)
#   D: ASVI4 instead of ASVI8 (4.8.4)
specs = []
for ret, basket in [("mag7_ret", "Mag7"), ("mag7b_ret", "Mag7B"), ("mag7c_ret", "Mag7C")]:
    for flow in ["flow_spy", "flow_rsp", "flow_voo"]:
        specs.append(("A" if basket == "Mag7" else "B", basket, flow[5:].upper(),
                      ["dlog_vix", "d_dgs10", "ASVI8", flow, ret]))
specs.append(("C", "Mag7, 3-variable", "None", ["dlog_vix", "ASVI8", "mag7_ret"]))
specs.append(("D", "Mag7, ASVI4", "SPY", ["dlog_vix", "d_dgs10", "ASVI4", "flow_spy", "mag7_ret"]))

results = {}
rows = []
for name, data in samples.items():
    for group, basket, flow, c in specs:
        r = VAR(data[c]).fit(1)
        results[(basket, flow, name)] = r
        att = c[2] if len(c) == 5 else c[1]            # attention variable
        fl = c[3] if len(c) == 5 else None             # flow (none in the three-variable VAR)
        ret = c[-1]                                    # return variable
        f, p = granger(r, att, ret)
        w, n = r.test_whiteness(nlags=10), r.test_normality()
        rows.append({
            "group": group, "basket": basket, "flow": flow, "sample": name, "N": int(r.nobs),
            "beta": r.params.loc["L1." + att, ret], "se": r.stderr.loc["L1." + att, ret],
            "beta_p": r.pvalues.loc["L1." + att, ret], "F": f, "p": p,
            "p_att_to_flow": granger(r, att, fl)[1] if fl else np.nan,
            "p_flow_to_ret": granger(r, fl, ret)[1] if fl else np.nan,
            "p_ret_to_att": granger(r, ret, att)[1],
            "fevd_att": fevd(r, ret).loc[att, "h=12"],
            "fevd_flow": fevd(r, ret).loc[fl, "h=12"] if fl else np.nan,
            "bic_lag": int(VAR(data[c]).select_order(maxlags=4).selected_orders["bic"]),
            "max_eig": largest_eigenvalue(r), "Q": w.test_statistic, "Q_p": w.pvalue,
            "normality": n.test_statistic, "normality_p": n.pvalue,
            "own_lag_att": r.params.loc["L1." + att, att],
        })
specs = pd.DataFrame(rows)
asvi8 = specs[specs["basket"] != "Mag7, ASVI4"]
print(len(specs), "VARs estimated")

22 VARs estimated


In [15]:
# Table 6
def spec_label(row):
    if row.group == "A":
        return f"{row.flow}, {row['sample']}"
    if row.group == "B":
        return f"{row.basket} ({'excl. TSLA' if row.basket == 'Mag7B' else 'AI-pure'}), {row['sample']}"
    if row.group == "C":
        return f"3-var, {row['sample']}"
    return f"ASVI4, SPY, {row['sample']}"

t6 = specs[specs["group"].isin(["A", "C", "D"]) | ((specs["group"] == "B") & (specs["flow"] == "SPY"))].copy()
t6["s"] = t6["sample"].map({"Full sample": 0, "2021-2025": 1})
t6["b"] = t6["basket"].map({"Mag7": 0, "Mag7B": 1, "Mag7C": 2}).fillna(0)
t6["f"] = t6["flow"].map({"SPY": 0, "RSP": 1, "VOO": 2, "None": 0})
t6 = t6.sort_values(["group", "s", "b", "f"])           # row order of Table 6

table6 = pd.DataFrame({
    "Panel": t6["group"].values,
    "Specification": [spec_label(r) for _, r in t6.iterrows()],
    "beta": [f"{x:.4f}" for x in t6["beta"]],
    "SE": [f"({x:.4f})" for x in t6["se"]],
    "Granger F": [f"{f:.3f}{stars(p)}" for f, p in zip(t6["F"], t6["p"])],
    "p": [f"{x:.3f}" for x in t6["p"]],
    "FEVD h=12": [f"{x:.1f}%" for x in t6["fevd_att"]],
    "N": t6["N"].values})
table6.to_csv("output/Table_6_robustness_summary.csv", index=False)
print("ASVI8 specifications significant at 5%:", (asvi8["p"] < 0.05).sum(), "of", len(asvi8))
table6

ASVI8 specifications significant at 5%: 20 of 20


,Panel,Specification,beta,SE,Granger F,p,FEVD h=12,N
0,A,"SPY, Full sample",0.0274,(0.0128),4.625**,0.032,4.8%,120
1,A,"RSP, Full sample",0.0269,(0.0127),4.491**,0.035,4.8%,120
2,A,"VOO, Full sample",0.0274,(0.0127),4.659**,0.031,4.8%,120
3,A,"SPY, 2021-2025",0.0378,(0.0161),5.500**,0.020,10.9%,60
4,A,"RSP, 2021-2025",0.0385,(0.0160),5.833**,0.016,11.1%,60
5,A,"VOO, 2021-2025",0.0397,(0.0157),6.384**,0.012,10.7%,60
6,B,"Mag7B (excl. TSLA), Full sample",0.0306,(0.0118),6.682***,0.010,6.1%,120
7,B,"Mag7C (AI-pure), Full sample",0.0349,(0.0127),7.549***,0.006,8.0%,120
8,B,"Mag7B (excl. TSLA), 2021-2025",0.0394,(0.0153),6.611**,0.011,13.3%,60
9,B,"Mag7C (AI-pure), 2021-2025",0.0413,(0.0172),5.723**,0.017,14.6%,60


In [16]:
# Table B.2: diagnostics of all specifications
tableB2 = pd.DataFrame({
    "Sample": specs["sample"], "Specification": specs["basket"], "Flow": specs["flow"],
    "BIC lag": specs["bic_lag"],
    "Max. eigenvalue modulus": [f"{x:.3f}" for x in specs["max_eig"]],
    "Portmanteau Q": [f"{x:.1f}" for x in specs["Q"]],
    "p (Q)": [f"{x:.3f}" for x in specs["Q_p"]],
    "Normality": [f"{x:.2f}" for x in specs["normality"]],
    "p (normality)": ["< 0.001" if x < 0.001 else f"{x:.3f}" for x in specs["normality_p"]]})
tableB2.to_csv("output/Table_B2_diagnostics_all_specifications.csv", index=False)
print("ASVI8 specifications: stable", (asvi8["max_eig"] < 1).sum(), "| no residual autocorrelation at 5%",
      (asvi8["Q_p"] > 0.05).sum(), "| BIC lag 1", (asvi8["bic_lag"] == 1).sum(),
      "| normality not rejected", (asvi8["normality_p"] > 0.05).sum(), "(of 20)")
tableB2

ASVI8 specifications: stable 20 | no residual autocorrelation at 5% 19 | BIC lag 1 20 | normality not rejected 4 (of 20)


,Sample,Specification,Flow,BIC lag,Max. eigenvalue modulus,Portmanteau Q,p (Q),Normality,p (normality)
0,Full sample,Mag7,SPY,1,0.793,234.9,0.312,57.65,< 0.001
1,Full sample,Mag7,RSP,1,0.789,241.0,0.221,110.24,< 0.001
2,Full sample,Mag7,VOO,1,0.791,240.1,0.233,29.02,0.001
3,Full sample,Mag7B,SPY,1,0.794,229.2,0.410,59.46,< 0.001
4,Full sample,Mag7B,RSP,1,0.791,231.0,0.378,112.55,< 0.001
5,Full sample,Mag7B,VOO,1,0.792,238.2,0.260,29.18,0.001
6,Full sample,Mag7C,SPY,1,0.793,228.2,0.428,59.44,< 0.001
7,Full sample,Mag7C,RSP,1,0.790,236.9,0.280,112.28,< 0.001
8,Full sample,Mag7C,VOO,1,0.791,241.0,0.221,28.55,0.001
9,Full sample,"Mag7, 3-variable",None,1,0.793,108.5,0.023,27.81,< 0.001


In [17]:
# Table C.3: Granger tests and variance decomposition of all specifications
def blank_if_missing(x, d):
    return "-" if pd.isna(x) else f"{x:.{d}f}"

tableC3 = pd.DataFrame({
    "Sample": specs["sample"], "Specification": specs["basket"], "Flow": specs["flow"],
    "beta on L1.ASVI (SE)": [f"{b:.4f}{stars(p)} ({s:.4f})" for b, s, p in zip(specs["beta"], specs["se"], specs["beta_p"])],
    "F": [f"{x:.3f}" for x in specs["F"]],
    "p": [f"{x:.3f}" for x in specs["p"]],
    "p: ASVI to flow": [blank_if_missing(x, 3) for x in specs["p_att_to_flow"]],
    "p: flow to return": [blank_if_missing(x, 3) for x in specs["p_flow_to_ret"]],
    "p: return to ASVI": [f"{x:.3f}" for x in specs["p_ret_to_att"]],
    "FEVD ASVI (%)": [f"{x:.1f}" for x in specs["fevd_att"]],
    "FEVD flow (%)": [blank_if_missing(x, 1) for x in specs["fevd_flow"]]})
tableC3.to_csv("output/Table_C3_granger_fevd_all_specifications.csv", index=False)

# Section 5.1: the 18 specifications with a flow variable and ASVI8
with_flow = asvi8[asvi8["flow"] != "None"]
print(f"ASVI8 -> flow p-values: {with_flow['p_att_to_flow'].min():.3f} to {with_flow['p_att_to_flow'].max():.3f}, "
      f"flow -> return significant at 5% in {(with_flow['p_flow_to_ret'] < 0.05).sum()} of {len(with_flow)}")
tableC3

ASVI8 -> flow p-values: 0.200 to 0.994, flow -> return significant at 5% in 0 of 18


,Sample,Specification,Flow,beta on L1.ASVI (SE),F,p,p: ASVI to flow,p: flow to return,p: return to ASVI,FEVD ASVI (%),FEVD flow (%)
0,Full sample,Mag7,SPY,0.0274** (0.0128),4.625,0.032,0.210,0.976,0.192,4.8,0.5
1,Full sample,Mag7,RSP,0.0269** (0.0127),4.491,0.035,0.913,0.483,0.161,4.8,0.5
2,Full sample,Mag7,VOO,0.0274** (0.0127),4.659,0.031,0.925,0.765,0.163,4.8,1.2
3,Full sample,Mag7B,SPY,0.0306*** (0.0118),6.682,0.010,0.219,0.984,0.202,6.1,0.3
4,Full sample,Mag7B,RSP,0.0304*** (0.0118),6.656,0.010,0.959,0.720,0.185,6.2,0.5
5,Full sample,Mag7B,VOO,0.0305*** (0.0118),6.732,0.010,0.884,0.611,0.154,5.9,2.3
6,Full sample,Mag7C,SPY,0.0349*** (0.0127),7.549,0.006,0.200,0.942,0.099,8.0,0.3
7,Full sample,Mag7C,RSP,0.0348*** (0.0127),7.558,0.006,0.936,0.984,0.089,8.2,0.5
8,Full sample,Mag7C,VOO,0.0347*** (0.0126),7.558,0.006,0.774,0.558,0.072,7.7,2.2
9,Full sample,"Mag7, 3-variable",None,0.0275** (0.0126),4.778,0.029,-,-,0.144,5.2,-


In [18]:
# Table C.4: variance decomposition of Mag7 returns in VAR-RSP and VAR-VOO (%)
tableC4 = pd.concat({
    f"VAR-{flow}": pd.concat({name: fevd(results[("Mag7", flow, name)], "mag7_ret").rename(index=labels)
                              for name in samples}, axis=1)
    for flow in ["RSP", "VOO"]}).round(1)
tableC4.to_csv("output/Table_C4_FEVD_RSP_VOO.csv")
tableC4

Full sample             2021-2025            
                             h=1   h=6  h=12       h=1   h=6  h=12
VAR-RSP dlogVIX             28.0  27.1  27.0      24.9  23.3  23.1
        dDGS10               2.2   2.3   2.3       7.5   6.9   6.8
        ASVI8                1.9   4.6   4.8       3.3  10.1  11.1
        Flow RSP             0.0   0.5   0.5       0.0   0.4   0.5
        Mag7 returns        67.8  65.5  65.3      64.4  59.3  58.5
VAR-VOO dlogVIX             28.2  27.3  27.3      22.3  21.7  21.7
        dDGS10               2.3   2.4   2.4       7.7   7.0   6.9
        ASVI8                1.9   4.5   4.8       4.8  10.0  10.7
        Flow VOO             1.0   1.2   1.2       0.8   3.4   3.5
        Mag7 returns        66.5  64.6  64.4      64.3  57.9  57.1

In [19]:
# Numbers in Sections 4.8.2 to 4.8.4
spy = specs[(specs["flow"] == "SPY") & specs["basket"].isin(["Mag7", "Mag7B", "Mag7C"])]
print(spy.pivot(index="basket", columns="sample", values=["beta", "F", "fevd_att"]).round(3))

mag7c = specs[(specs["basket"] == "Mag7C") & (specs["sample"] == "Full sample")]
print("\nMag7C, full sample, return -> ASVI8 p:", dict(zip(mag7c["flow"], mag7c["p_ret_to_att"].round(3))))

three = specs[specs["basket"] == "Mag7, 3-variable"].set_index("sample")
five = specs[(specs["basket"] == "Mag7") & (specs["flow"] == "SPY")].set_index("sample")
print("\n3-variable VAR: beta", three["beta"].round(4).to_dict(), "| whiteness p", three["Q_p"].round(3).to_dict())
print("Largest difference to the 5-variable beta:", round((three["beta"] - five["beta"]).abs().max(), 4))

a4 = specs[specs["basket"] == "Mag7, ASVI4"].set_index("sample")
print("\nOwn lag, full sample: ASVI4", round(a4.loc["Full sample", "own_lag_att"], 3),
      "| ASVI8", round(five.loc["Full sample", "own_lag_att"], 3))

            beta                     F              fevd_att            
sample 2021-2025 Full sample 2021-2025 Full sample 2021-2025 Full sample
basket                                                                  
Mag7       0.038       0.027     5.500       4.625    10.937       4.830
Mag7B      0.039       0.031     6.611       6.682    13.304       6.108
Mag7C      0.041       0.035     5.723       7.549    14.566       7.952

Mag7C, full sample, return -> ASVI8 p: {'SPY': 0.099, 'RSP': 0.089, 'VOO': 0.072}

3-variable VAR: beta {'Full sample': 0.0275, '2021-2025': 0.0386} | whiteness p {'Full sample': 0.023, '2021-2025': 0.131}
Largest difference to the 5-variable beta: 0.0008

Own lag, full sample: ASVI4 0.627 | ASVI8 0.771


In [20]:
# Table C.7: Cholesky ordering. The alternative puts ASVI8 last, so attention cannot move
# any other variable within the same month.
orderings = {"Baseline": ["dlog_vix", "d_dgs10", "ASVI8", "flow_spy", "mag7_ret"],
             "ASVI8 last": ["dlog_vix", "d_dgs10", "flow_spy", "mag7_ret", "ASVI8"]}
rows = []
for name, data in samples.items():
    for label, c in orderings.items():
        r = VAR(data[c]).fit(1)
        fe = fevd(r, "mag7_ret").loc["ASVI8"]
        point, lower, upper = irf_bands(r, "ASVI8", "mag7_ret")
        rows.append({"Sample": name, "Ordering": label,
                     "FEVD h=1 (%)": f"{fe['h=1']:.1f}", "FEVD h=6 (%)": f"{fe['h=6']:.1f}",
                     "FEVD h=12 (%)": f"{fe['h=12']:.1f}", "IRF h=0 (%)": f"{point[0] * 100:.3f}",
                     "IRF h=1 (%) [95% CI]": f"{point[1] * 100:.3f} [{lower[1] * 100:.3f}, {upper[1] * 100:.3f}]"})
tableC7 = pd.DataFrame(rows)
tableC7.to_csv("output/Table_C7_Cholesky_ordering.csv", index=False)
tableC7

,Sample,Ordering,FEVD h=1 (%),FEVD h=6 (%),FEVD h=12 (%),IRF h=0 (%),IRF h=1 (%) [95% CI]
0,Full sample,Baseline,1.9,4.6,4.8,1.008,"0.802 [-0.033, 1.637]"
1,Full sample,ASVI8 last,0.0,3.0,3.3,0.000,"0.915 [0.073, 1.757]"
2,2021-2025,Baseline,2.8,9.8,10.9,1.305,"1.374 [0.198, 2.551]"
3,2021-2025,ASVI8 last,0.0,6.9,7.9,0.000,"1.395 [0.203, 2.587]"
